In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import requests

In [2]:
df = pd.read_pickle(r"E:\Data Science\Projects\AI Teaching Assistant\new_embeddings.pkl")

OLLAMA_URL = "http://localhost:11434/api/embed"
MODEL_NAME = "bge-m3"
BATCH_SIZE = 16

In [3]:
def create_embedding(text_list):
    r = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "input": text_list
        },
        timeout=300
    )

    r.raise_for_status()

    response = r.json()

    if "embeddings" not in response:
        raise Exception(
            f"Embeddings not found in response.\nResponse: {response}"
        )

    return response["embeddings"]

In [4]:
question = input("Enter your question: ")
question_embedding = create_embedding([question])[0]

In [5]:
similarities = cosine_similarity(np.vstack(df['embedding'].values), [question_embedding]).flatten()
print(f"For question '{question}', similarities with existing embeddings: {similarities}")
print(f"Indices of top 5 most similar embeddings: {similarities.argsort()[-5:][::-1]}")  # Print indices of top 5 most similar embeddings

For question 'How does weights and biases work?', similarities with existing embeddings: [0.33047001 0.3031368  0.30728384 ... 0.4179354  0.42084373 0.3500042 ]
Indices of top 5 most similar embeddings: [1582 2508 1572 2376 2409]


In [6]:
df.iloc[similarities.argsort()[-5:][::-1]]  # Display the top 5 most similar embeddings

,number,title,start,end,text,chunk_id,embedding
1582,10,Forward Propagation | How a neural network pre...,340.0,350.0,of weights and biases. Now let's see how the ...,1582,"[-0.01580258, 0.027636958, -0.0456165, 0.02972..."
2508,15,Backpropagation in Deep Learning | Part 1 | Th...,1177.0,1182.0,These Weights And Bias So,2508,"[-0.013451047, 0.014376534, -0.054354765, -0.0..."
1572,10,Forward Propagation | How a neural network pre...,240.0,250.0,When the back propagation algorithm works it ...,1572,"[-0.036520593, -0.023666156, -0.05104014, -0.0..."
2376,15,Backpropagation in Deep Learning | Part 1 | Th...,301.0,310.0,Okay Now what happens is There are two things...,2376,"[-0.00550509, -0.006778721, -0.052332547, 0.03..."
2409,15,Backpropagation in Deep Learning | Part 1 | Th...,627.0,634.0,Here all the weights Their value Will be 1 An...,2409,"[-8.252722e-06, 0.008083004, -0.06290854, 0.00..."


In [7]:
new_df = df.iloc[similarities.argsort()[-5:][::-1]]
dff = new_df[['number', 'title', 'start', 'end', 'text']]

In [8]:
prompt = f"""
You are an AI assistant helping students navigate a Deep Learning course.

Below are video transcript chunks. Each chunk contains:
- video_number
- video_title
- start (seconds)
- end (seconds)
- transcript text

Video Chunks:
{dff.to_json(orient="records")}

--------------------------------------------------

Student Question:
"{question}"

Your task:

1. Find all transcript chunks that are relevant to the student's question.
2. Match using semantic meaning, not just exact keyword matching.
   - Include synonyms and related concepts whenever appropriate.
3. Group results by video.
4. For each relevant video, provide:
   - Video Number
   - Video Title
   - Timestamp in MM:SS format
   - A very short description (5–15 words) of what is explained there.
5. Rank results from most relevant to least relevant.
6. If multiple nearby chunks belong to the same explanation, merge them into one timestamp range.
7. If only a brief mention exists, explicitly state "Brief mention".
8. If a concept is explained in depth across multiple timestamps in the same video, mention each important section.

Output format:

Video <number> – <title>
• MM:SS–MM:SS — <short description>

Video <number> – <title>
• MM:SS — Brief mention

Rules:
- Keep the response under 120 words.
- Do not use end time in the output.
- Do not invent timestamps or explanations.
- Only use the provided transcript chunks.
- Convert starting seconds to MM:SS format.
- If no relevant content exists, respond exactly:
No content found
- Do not ask follow-up questions.
- Do not add introductory or concluding text.
"""

promptt = f"""
Video transcript chunks:
{dff.to_json(orient="records")}

Question: "{question}"

Find all chunks relevant to the question (semantic match, not just keywords).

Return only:
The Relevant Videos for your question, "{question}":
Video <number> - <title>
• MM:SS : <A short enhanced description>

Group nearby chunks from the same video. Sort by relevance. Convert starting seconds to MM:SS. Use only the given data. If no relevant chunk exists, reply exactly:
No content found, do not use words like chunk, section, or part instead use word "video". Do not invent timestamps or explanations.
"""

In [9]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

def get_api_key():
    """Get API key from environment variables"""
    # Try GEMINI_API_KEY first
    api_key = os.getenv("GEMINI_API_KEY")
    if api_key:
        return api_key
    return None

api_key = get_api_key()
if not api_key:
    raise ValueError("API key not found. Please check your .env file.")

C:\Users\shree\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash') 
model.start_chat(history=[])
# Send message to the model
response = model.generate_content(prompt)
print(response.text)

Video 10 – Forward Propagation | How a neural network predicts output
• 04:00 — Backpropagation determines the values of weights and biases.
• 05:40 — Mentions weights and biases are used when a neural network predicts output.

Video 15 – Backpropagation in Deep Learning | Part 1 | The What
• 05:01 — Identifies weights and biases as two core components of a neural network.
• 10:27 — Brief example demonstrating initial values for weights and biases.
• 19:37 — Brief mention.
